In [1]:

from loguru import logger
# logger.disable(None)
logger.disable("nymeria")
logger.disable("projectaria_tools")
logger.disable("VrsDataProvider")
logger.disable("MpsDataPathsProvider")
logger.disable("ProgressLogger")
logger.disable("MultiRecordFileReader")

import shutil
from nymeria.download_utils import DownloadManager
from nymeria.definitions import DataGroups
from nymeria.data_provider import SequencePathProvider, NymeriaDataProvider
from nymeria.definitions import Subpaths, VrsFiles
from nymeria.recording_data_provider import create_recording_data_provider
from projectaria_tools.core import data_provider
from projectaria_tools.core import sophus
from projectaria_tools.core.sensor_data import TimeDomain

import torch
import numpy as np
from PIL import Image
from PIL import ImageDraw
from pathlib import Path
import os
import yaml 
from torchvision import transforms
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler

import sys
sys.path.append("/home/anw2067/visualnav-transformer/train")
from vint_train.models.nomad.conditional_uned1dnomad import ConditionalUnet1D_NoMaD
from vint_train.data.misc import XSensConstants, XsensSkeleton
from vint_train.models.nomad.nomad import DenseNetwork, NoMaD
from vint_train.models.nomad.nomad_vint import NoMaD_ViNT, replace_bn_with_gn
from vint_train.training.nymeria_training_utils import unnormalize_data_smpl_pose_gaussian, forward_kinematics_wrapper
from vint_train.training.train_eval_loop import (
    
    load_model as load_model_from_ckpt
)
from vint_train.training.nymeria_training_utils import get_action_smpl_torch
from vint_train.data.vint_dataset import ViNT_Nymeria_Dataset
from vint_train.training.train_utils import model_output

MODEL_DIR = "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw"
CONFIG_PATH = "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/config.yaml"
TEST_TRACKS = "/home/anw2067/visualnav-transformer/train/data_splits/nymeria/train/traj_names.txt"
DATA_JSON="/home/anw2067/visualnav-transformer/data_jsons/visibility_no_data.json"
DATA_SAVE_DIR = "/home/anw2067/scratch/nymeria_test_dir"
VIS_OUTPUT_DIR = "/home/anw2067/visualnav-transformer/train/logs/visualizations"

config = yaml.load(open(CONFIG_PATH), Loader=yaml.FullLoader)



/scratch/anw2067/conda/envs/nomad_train/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch/anw2067/conda/envs/nomad_train/lib/python3.10/site-packages/wandb/sdk/internal/internal_api.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [2]:
def download_episode(json_path, save_dir, ep):
    """Download a single episode - must be a top-level function for multiprocessing"""
    dl = DownloadManager(Path(json_path), out_rootdir=Path(save_dir))
    dl.download(match_key=ep, selected_groups=[DataGroups.recording_head, DataGroups.body_motion], ignore_existing=True)
    # remove possibly full dir os.path.join(save_dir, ep, "recording_head", "mps")
    # if os.path.exists(os.path.join(save_dir, ep, "recording_head", "mps")):
    #     shutil.rmtree(os.path.join(save_dir, ep, "recording_head", "mps"))
    # if os.path.exists(os.path.join(save_dir, ep, "recording_head", "data", "et.vrs")):
    #     os.remove(os.path.join(save_dir, ep, "recording_head", "data", "et.vrs"))
    return ep

In [3]:
def load_camera_model(path, ep):
    path = Path(os.path.join(path, ep))
    seq_pd = SequencePathProvider(path)
    vrs_dp = data_provider.create_vrs_data_provider(str(seq_pd.recording_head / VrsFiles.motion))
    return vrs_dp.get_device_calibration().get_camera_calib("camera-rgb")

In [4]:
def load_model(config, model_dir, device):
    def get_vision_encoder():
        if config.get("goal_type", None) in ["2d", "2d5050"]:
            goal_coordinate_dims = 8
        else:
            goal_coordinate_dims = 0
        vision_encoder = NoMaD_ViNT(
            obs_encoder=config["obs_encoder"],
            obs_encoding_size=config["encoding_size"],
            context_size=config["context_size"],
            mha_num_attention_heads=config["mha_num_attention_heads"],
            mha_num_attention_layers=config["mha_num_attention_layers"],
            mha_ff_dim_factor=config["mha_ff_dim_factor"],
            pool_features=config.get("pool_features", True),
            image_size=config["image_size"],
            proprioception=config.get("proprioception", False),
            project_encoding=config.get("project_encoding", False),
            pos_enc_3d=config.get("pos_enc_3d", False),
            pool_curr_obs=config.get("pool_curr_obs", False),
            goal_coordinate_dims=goal_coordinate_dims,
        )
        vision_encoder = replace_bn_with_gn(vision_encoder)
        return vision_encoder
    # Create the model
    vision_encoder = get_vision_encoder()
    
    if config.get("goal_type", None) == "cheat":
        goal_pose_dim = 48
    elif config.get("goal_type", None) == "point":
        goal_pose_dim = 4 * 3 # 3 dimensions each for (Head, LHand, RHand, Pelvis)
    else:
        goal_pose_dim = 0
    noise_pred_net = ConditionalUnet1D_NoMaD(input_dim=config['input_dims'],
                                                global_cond_dim=config["encoding_size"],
                                                down_dims=config["down_dims"],
                                                cond_predict_scale=config["cond_predict_scale"],
                                                goal_pose_dims=goal_pose_dim)
    dist_pred_network = DenseNetwork(embedding_dim=config["encoding_size"])
    model = NoMaD(vision_encoder, noise_pred_net, dist_pred_network)
    noise_scheduler = DDPMScheduler(num_train_timesteps=config["num_diffusion_iters"], beta_schedule='squaredcos_cap_v2', clip_sample=True, prediction_type='epsilon')
    
    model = model.to(device)
    latest_path = os.path.join(model_dir, "latest.pth")
    if os.path.exists(latest_path):
        latest_checkpoint = torch.load(latest_path, map_location=device)
        no_module_keys_dict = {k.replace('module.', ''): v for k, v in latest_checkpoint.items() if 'module' in k}
        msg = load_model_from_ckpt(model, config["model_type"], no_module_keys_dict)  # Use model.module for DDP
        print(msg)
    return model, noise_scheduler

In [5]:
def get_dataset(config, dataset_split_type):
    transform = ([
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    transform = transforms.Compose(transform)
    
    data_config = config["datasets"]["nymeria"]
    data_config[dataset_split_type] = "/home/anw2067/visualnav-transformer/train/data_splits/nymeria/test"
    data_config['gaussian_normalization_stats_path'] = "/home/anw2067/visualnav-transformer/train/nymeria_nomad_mean_var_stats.json"
    
    dataset = ViNT_Nymeria_Dataset(
        data_folder=data_config["data_folder"],
        data_split_folder=data_config[dataset_split_type],
        dataset_name="nymeria",
        image_size=config["image_size"],
        transform=transform,
        waypoint_spacing=1,
        min_dist_cat=config["distance"]["min_dist_cat"],
        max_dist_cat=config["distance"]["max_dist_cat"],
        min_action_distance=config["action"]["min_dist_cat"],
        max_action_distance=config["action"]["max_dist_cat"],
        negative_goals=False,
        len_traj_pred=config["len_traj_pred"],
        context_size=config["context_size"],
        goal_type=config.get("goal_type", None),
        preserve_pose_up_down=data_config.get("preserve_pose_up_down", False),
        context_type=config["context_type"],
        end_slack=0,
        goals_per_obs=1,
        normalize=config["normalize"],
        gaussian_normalization_stats_path=data_config["gaussian_normalization_stats_path"],
    )
    return dataset

In [6]:
def pose_to_image_coords(pose, cam_model, xsens_offsets, T_C_pelvis, image_size=224) -> torch.Tensor:
    """
    pose: B, 48
    cam_model: CameraCalibration
    xsens_offsets: 23, 3
    T_C_pelvis: Sophus SE3
    
    Returns:
        image_coords: B, 15, 2
    """
    device = pose.device
    xsens_skel = XsensSkeleton(offsets=xsens_offsets)
    pose_xyz, pose_rpy = forward_kinematics_wrapper(pose, xsens_skel, XSensConstants.upper_body_num_parts, return_euler=True) # B, 15, 3
    
    R_C_pelvis = torch.from_numpy(T_C_pelvis.rotation().to_matrix()).to(device, dtype=pose.dtype)
    t_C_pelvis = torch.from_numpy(T_C_pelvis.translation()).to(device, dtype=pose.dtype)
    
    # Ensure t_C_pelvis is the right shape: (3,) or (1, 3) -> (3,)
    if t_C_pelvis.dim() > 1:
        t_C_pelvis = t_C_pelvis.squeeze(0)
    
    # convert everything to camera frame
    # pose_xyz: (B, 15, 3) - 15 body parts, each with 3D coordinates
    # R_C_pelvis: (3, 3) - rotation matrix from pelvis frame to camera frame
    # t_C_pelvis: (3,) - translation vector from pelvis frame to camera frame
    # Transform: R_C_pelvis @ p + t_C_pelvis for each point p
    # Use matmul to transform each 3D point: (B, 15, 3) @ (3, 3).T -> (B, 15, 3)
    pose_xyz_cam = (torch.matmul(pose_xyz, R_C_pelvis.T) + t_C_pelvis).detach().cpu().numpy().astype(np.float64) # B, 15, 3
    
    res = []
    for b in range(pose.shape[0]):
        res.append([])
        for i in range(pose_xyz_cam.shape[1]):
            coords = cam_model.project(pose_xyz_cam[b, i, :, None])
            if coords is not None:
                rotated_coords = np.empty_like(coords)
                rotated_coords[0] = 2880 - 1 - coords[1]
                rotated_coords[1] = coords[0]
                rotated_coords = rotated_coords / 2880 * image_size
                res[b].append(torch.tensor(rotated_coords))
            else:
                res[b].append(torch.tensor([-1, -1]))
        res[b] = torch.stack(res[b], dim=0)
    image_coords = torch.stack(res, dim=0)
    return image_coords
    
    
def get_T_C_pelvis(nymeria_dp, index):
    """
    nymeria_dp: NymeriaDataProvider
    index: int
    """
    start_ns, end_ns = nymeria_dp._NymeriaDataProvider__get_timespan_ns()
    curr_ns = start_ns + index * 0.25 * 1e9
    
    T_C_Hd = nymeria_dp.recording_head.vrs_dp.get_device_calibration().get_camera_calib("camera-rgb").get_transform_device_camera().inverse()
    T_Hd_Wd = nymeria_dp.recording_head.get_pose(curr_ns, time_domain=TimeDomain.TIME_CODE)[0].transform_world_device.inverse()
    
    # get pelvis-wd
    data = nymeria_dp.get_synced_poses(curr_ns, return_orientation=True)
    parts = data['parts']
    pelvis = parts[XSensConstants.part_names.index("Pelvis"), 0]
    T_Wd_P = sophus.SE3.from_quat_and_translation(pelvis[:1, None], pelvis[1:4, None], pelvis[4:, None])
    
    return T_C_Hd @ T_Hd_Wd @ T_Wd_P

def draw_image_coords(draw, goal_image_coords, color=(255, 255, 255), num_segments=XSensConstants.upper_body_num_parts, show_text=True):
    num_visible = 0
    for part_name in XSensConstants.part_names[:num_segments]:
        index = XSensConstants.part_names.index(part_name)
        parent_index = XSensConstants.kintree_parents[index]

        point = goal_image_coords[0, index]
        if all(point == -1):
            continue
        num_visible += 1
        
        draw.ellipse([point[0]-3, point[1]-3, point[0]+3, point[1]+3], fill=color)
        if any(x in part_name for x in ["Pelvis", "Head", "Hand"]) and show_text:
            draw.text((point[0], point[1]), part_name, fill=color)
        if parent_index != -1:
            parent_point = goal_image_coords[0, parent_index]
            if not all(parent_point == -1):
                draw.line([*point, *parent_point], fill=color)
    return int(num_visible)

def get_num_visible(goal_image_coords, num_segments=XSensConstants.upper_body_num_parts):
    return int((1.-(goal_image_coords[0, :num_segments] == -1).all(-1).float()).sum())


In [ ]:
def draw_forward_kinematics_error(dataset, device):
    for idx in range(0, len(dataset), 100): # every 100 steps, i.e. 25 seconds
        track_name, track_idx, _ = dataset.index_to_data[idx]
        print(f"{idx}/{len(dataset)} Processing {track_name} at {track_idx} steps")
    
        data_dict = dataset[idx]
        goal_image_coords = data_dict["goal_image_coords"][None].to(device, non_blocking=True) # B, 23, 2
        xsens_offsets = data_dict["xsens_offsets"].to(device, non_blocking=True)
        vis_image = data_dict["obs_images"][-1]
        image = Image.fromarray((255.*vis_image.permute(1, 2, 0)).to(torch.uint8).numpy())
        draw = ImageDraw.Draw(image)
        
        if not os.path.exists(os.path.join(DATA_SAVE_DIR, track_name)):
            os.makedirs(os.path.join(DATA_SAVE_DIR, track_name))
            download_episode(DATA_JSON, DATA_SAVE_DIR, track_name)
        
        gt_actions_with_initial = data_dict["gt_actions_with_initial"][None].to(device, non_blocking=True)
        cam_model = load_camera_model(DATA_SAVE_DIR, track_name)
        nymeria_dp = NymeriaDataProvider(sequence_rootdir=Path(os.path.join(DATA_SAVE_DIR, track_name)), load_wrist=False, load_observer=False)
        T_C_Pelvis = get_T_C_pelvis(nymeria_dp, track_idx)
        gt_actions_with_initial_image_coords = pose_to_image_coords(gt_actions_with_initial[:, 0], cam_model, xsens_offsets, T_C_Pelvis) # B, 15, 2
        
        num_visible = draw_image_coords(draw, goal_image_coords, color=(0, 180, 180))
        num_visible = max(num_visible, draw_image_coords(draw, gt_actions_with_initial_image_coords, color=(180, 64, 64)))
        if num_visible > 0:
            image.save(os.path.join(VIS_OUTPUT_DIR, "forward_kinematics_error", f"{track_name}_{track_idx}-vis{num_visible}.png"))

@torch.no_grad()
def main():
    config = yaml.load(open(CONFIG_PATH), Loader=yaml.FullLoader)
    device = "cuda"
    model, noise_scheduler = load_model(config, MODEL_DIR, device)
    dataset = get_dataset(config, "test")
    
    prev_track_name = None
    for idx in range(0, len(dataset), 100): # every 100 steps, i.e. 25 seconds
        track_name, track_idx, _ = dataset.index_to_data[idx]
        print(f"{idx}/{len(dataset)} Processing {track_name} at {track_idx} steps")
    
        data_dict = dataset[idx]
        
        obs_image = data_dict["obs_image_transformed"][None].to(device)
        goal_image = data_dict["goal_image_transformed"][None].to(device)
        context_poses = data_dict["context_poses"][None].to(device)
        goal_image_coords = data_dict["goal_image_coords"][None].to(device, non_blocking=True) # B, 23, 2
        goal_pos = data_dict["goal_pos"][None].to(device, non_blocking=True)
        first_pose = data_dict["first_pose"][None].to(device, non_blocking=True)
        xsens_offsets = data_dict["xsens_offsets"].to(device, non_blocking=True)
        
        goal_coordinates = None
        if config.get("goal_type", None) in ["2d", "2d5050"]:
            goal_coordinates = torch.stack(
                [goal_image_coords[:, XSensConstants.part_names.index(part_name)] for part_name in ["Pelvis", "Head", "R_Hand", "L_Hand"]]
            , dim=1).flatten(1, 2) # B, 4*2
        if config.get("goal_type", None) == "point":
            goal_pos_xyz = data_dict["goal_pose_xyz"][None].to(device, non_blocking=True)[:, 0] # B, 15, 3
            goal_pose = torch.cat(
                [goal_pos_xyz[:, XSensConstants.part_names.index(part)] for part in ["Pelvis", "Head", "R_Hand", "L_Hand"]]
            , dim=-1) # B, 4*3
        else:
            # goal_pose = gt_actions_with_initial[:, 0]
            goal_pose = goal_pos[:, 0]
        
        
        pred_horizon = config["len_traj_pred"]
        action_dim = getattr(dataset, "num_action_params", 48)
        num_samples = 1
        B=1
        
        output = model_output(
            model,
            noise_scheduler,
            obs_image,
            goal_image,
            goal_pose,
            context_poses,
            pred_horizon,
            action_dim,
            num_samples,
            device,
            goal_coordinates,
        )
        
        gc_actions = unnormalize_data_smpl_pose_gaussian(
            output["gc_actions"].flatten(0, 1)
        ).unflatten(0, (B, -1))
        uc_actions = unnormalize_data_smpl_pose_gaussian(
            output["uc_actions"].flatten(0, 1)
        ).unflatten(0, (B, -1))
        
        gc_actions = get_action_smpl_torch(first_pose, gc_actions, XSensConstants.upper_body_num_parts) # B, T, 48
        uc_actions = get_action_smpl_torch(first_pose, uc_actions, XSensConstants.upper_body_num_parts) # B, T, 48
        
        if not os.path.exists(os.path.join(DATA_SAVE_DIR, track_name)):
            os.makedirs(os.path.join(DATA_SAVE_DIR, track_name))
            download_episode(DATA_JSON, DATA_SAVE_DIR, track_name)
            
        if track_name != prev_track_name:
            nymeria_dp = NymeriaDataProvider(sequence_rootdir=Path(os.path.join(DATA_SAVE_DIR, track_name)), load_wrist=False, load_observer=False)
            cam_model = load_camera_model(DATA_SAVE_DIR, track_name)
        prev_track_name = track_name
            
        T_C_Pelvis = get_T_C_pelvis(nymeria_dp, track_idx)
        
        vis_image = data_dict["goal_image"]
        image = Image.fromarray((255.*vis_image.permute(1, 2, 0)).to(torch.uint8).numpy())
        draw = ImageDraw.Draw(image)
        
        # gt_vis = draw_image_coords(draw, goal_image_coords, color=(0, 0, 255))
        gt_vis = get_num_visible(goal_image_coords)
        gc_vis = 0
        for i in [7, 4, 1]:
            color_multiplier = i / gc_actions.shape[1]
            # color_multiplier = (gc_actions.shape[1] - i) / gc_actions.shape[1] # (8-i) / 8
            
            gc_image_coords = pose_to_image_coords(gc_actions[:, i], cam_model, xsens_offsets, T_C_Pelvis) # B, 15, 2
            # uc_image_coords = pose_to_image_coords(uc_actions[:, i], cam_model, xsens_offsets, T_C_Pelvis) # B, 15, 2
            
            gc_vis_ = draw_image_coords(draw, gc_image_coords, color=(int(255*color_multiplier), int(255*color_multiplier), int(255*color_multiplier)), show_text=(i == 7))
            gc_vis = max(gc_vis, gc_vis_)
            # uc_vis = draw_image_coords(draw, uc_image_coords, color=(int(255*color_multiplier), 0, 0))
        if gt_vis > 0:
            if not os.path.exists(os.path.join(VIS_OUTPUT_DIR, "gc_pred_actions")):
                os.makedirs(os.path.join(VIS_OUTPUT_DIR, "gc_pred_actions"))
            image.save(os.path.join(VIS_OUTPUT_DIR, "gc_pred_actions", f"{track_name}_{track_idx}-gt{gt_vis}-gc{gc_vis}.png"))
            # goal_image.save(os.path.join(VIS_OUTPUT_DIR, "gc_pred_actions", f"{track_name}_{track_idx}-goal.png"))
        
main()

<All keys matched successfully>
0/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 3 steps
destination error tensor(-0.0083)


[ProgressLogger][INFO]: 2025-12-16 13:33:31: Opening /home/anw2067/visualnav-transformer/test_dir/20231207_s0_jodi_morrison_act4_iplp4x/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/visualnav-transformer/test_dir/20231207_s0_jodi_morrison_act4_iplp4x/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/home/anw2067/visualnav-transformer/test_dir/20231207_s0_jodi_morrison_act4_iplp4x/recording_head/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.
[MpsDataPathsProvider][WARNING]: Hand tracking

Loaded #closed loop trajectory poses records: 1062226


[ProgressLogger][INFO]: 2025-12-16 13:33:38: Opening /home/anw2067/visualnav-transformer/test_dir/20231207_s0_jodi_morrison_act4_iplp4x/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/visualnav-transformer/test_dir/20231207_s0_jodi_morrison_act4_iplp4x/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated


100/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 103 steps
destination error tensor(-0.0934)
200/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 203 steps
destination error tensor(0.0112)
300/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 303 steps
destination error tensor(-0.0484)
400/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 403 steps
destination error tensor(-0.0007)
500/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 503 steps
destination error tensor(0.1343)
600/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 603 steps
destination error tensor(-0.0066)
700/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 703 steps
destination error tensor(-0.0136)
800/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 803 steps
destination error tensor(-0.0464)
900/819302 Processing 20231207_s0_jodi_morrison_act4_iplp4x at 903 steps
destination error tensor(-0.0746)
1000/819302 Processing 20231207_s0_jodi

...

100%|██████████| 863M/863M [00:10<00:00, 79.9MiB/s] 
100%|██████████| 684M/684M [00:09<00:00, 69.4MiB/s] 
[ProgressLogger][INFO]: 2025-12-16 13:37:57: Opening /home/anw2067/visualnav-transformer/test_dir/20230607_s0_james_johnson_act1_7xwm28/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/visualnav-transformer/test_dir/20230607_s0_james_johnson_act1_7xwm28/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/home/anw2067/visualnav-transformer/test_dir/20230607_s0_james_johnson_act1_7xwm28/recording_head/mps/hand_tracking) does not 

Loaded #closed loop trajectory poses records: 997079


[ProgressLogger][INFO]: 2025-12-16 13:38:04: Opening /home/anw2067/visualnav-transformer/test_dir/20230607_s0_james_johnson_act1_7xwm28/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/visualnav-transformer/test_dir/20230607_s0_james_johnson_act1_7xwm28/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated


38700/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 122 steps
destination error tensor(0.0386)
38800/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 222 steps
destination error tensor(-0.0113)
38900/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 322 steps
destination error tensor(-0.0076)
39000/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 422 steps
destination error tensor(-0.0410)
39100/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 522 steps
destination error tensor(-0.0171)
39200/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 622 steps
destination error tensor(0.0028)
39300/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 722 steps
destination error tensor(0.0243)
39400/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 822 steps
destination error tensor(0.0328)
39500/819302 Processing 20230607_s0_james_johnson_act1_7xwm28 at 922 steps
destination error tensor(0.1713)
39600/819302 Processing 

OSError: [Errno 122] Disk quota exceeded: '/home/anw2067/visualnav-transformer/test_dir/20230828_s0_kaylee_johnson_act1_svrk6s'